In [96]:
from sympy import *
import string


In [97]:
x, t = symbols('x t')

H = Matrix([
    [1,0],
    [0,-1]
])

Ep = Matrix([
    [0,1],
    [0,0]
])

Em = Matrix([
    [0,0],
    [1,0]
])

v0 = Function('v0')(x, t)
A0 = v0*H

E = Ep + Em

In [98]:
num = int(input())
fields = {}
grade = 0
count = 0

for y in range(0, int(num+1+(num+1)/2)):
    print(y % 2)
    letter = string.ascii_lowercase[y]
    exec(f"{letter} = Function('{letter}')(x, t)")

    if grade not in fields:
        fields[grade] = []

    fields[grade].append(eval(letter))
    
    count += 1

    if grade % 2 == 0 and count == 1:
        grade += 1
        count = 0
    
    elif grade % 2 == 1 and count == 2:
        grade += 1
        count = 0
    

0
1
0
1
0
1
0
1
0


In [99]:

def comm(A,B):
    return A*B - B*A

In [100]:
grades = {}
fields[num][0] = 1
fields[num][1] = 1

for g in range(num, -1, -1):
    this_grade = zeros(2, 2) 

    if g != 0:
        if g % 2 != 0:
            D_n = (fields[g][0]*Ep + fields[g][1]*Em)
        else: 
            D_n = fields[g][0]*H 
        
        print(f"g is {g} and Dn is {D_n} ")

        if (g-1) % 2 != 0:
            D_n_minus_one = (fields[g-1][0]*Ep + fields[g-1][1]*Em)
        else: 
            D_n_minus_one = fields[g-1][0]*H 
        this_grade += diff(D_n, x) 
        this_grade += comm(E, D_n_minus_one) + comm(A0, D_n) 

    else:
        this_grade -= diff(A0, t)
        D_n = fields[g][0]*H
        this_grade += diff(D_n, x) + comm(A0, D_n)
    grades[g] = this_grade
    # diff(D_n, x)        
grades
    


g is 5 and Dn is Matrix([[0, 1], [1, 0]]) 
g is 4 and Dn is Matrix([[g(x, t), 0], [0, -g(x, t)]]) 
g is 3 and Dn is Matrix([[0, e(x, t)], [f(x, t), 0]]) 
g is 2 and Dn is Matrix([[d(x, t), 0], [0, -d(x, t)]]) 
g is 1 and Dn is Matrix([[0, b(x, t)], [c(x, t), 0]]) 


{5: Matrix([
 [                     0, -2*g(x, t) + 2*v0(x, t)],
 [2*g(x, t) - 2*v0(x, t),                       0]]),
 4: Matrix([
 [-e(x, t) + f(x, t) + Derivative(g(x, t), x),                                          0],
 [                                          0, e(x, t) - f(x, t) - Derivative(g(x, t), x)]]),
 3: Matrix([
 [                                                      0, -2*d(x, t) + 2*e(x, t)*v0(x, t) + Derivative(e(x, t), x)],
 [2*d(x, t) - 2*f(x, t)*v0(x, t) + Derivative(f(x, t), x),                                                        0]]),
 2: Matrix([
 [-b(x, t) + c(x, t) + Derivative(d(x, t), x),                                          0],
 [                                          0, b(x, t) - c(x, t) - Derivative(d(x, t), x)]]),
 1: Matrix([
 [                                                      0, -2*a(x, t) + 2*b(x, t)*v0(x, t) + Derivative(b(x, t), x)],
 [2*a(x, t) - 2*c(x, t)*v0(x, t) + Derivative(c(x, t), x),                                           

In [101]:
system = {}
for key, value in grades.items():
    if key % 2 != 0: # odd grade
        Eq_p = value[0,1]
        Eq_m = value[1,0]
        display(Eq_p)
        display(Eq_m)
        system[key] = [Eq_m, Eq_p]
    else: # even grade
        Eq_H = value[0,0]
        display(Eq_H)
        system[key] = [Eq_H]

# for key, value in system.items():
#     if key % 2 != 0: # odd grade
#         display
#     else: # even grade
#         Eq_H = value[0,0]
#         system[key] = [Eq_H]

-2*g(x, t) + 2*v0(x, t)

2*g(x, t) - 2*v0(x, t)

-e(x, t) + f(x, t) + Derivative(g(x, t), x)

-2*d(x, t) + 2*e(x, t)*v0(x, t) + Derivative(e(x, t), x)

2*d(x, t) - 2*f(x, t)*v0(x, t) + Derivative(f(x, t), x)

-b(x, t) + c(x, t) + Derivative(d(x, t), x)

-2*a(x, t) + 2*b(x, t)*v0(x, t) + Derivative(b(x, t), x)

2*a(x, t) - 2*c(x, t)*v0(x, t) + Derivative(c(x, t), x)

Derivative(a(x, t), x) - Derivative(v0(x, t), t)

In [ ]:
equations = []
is_first = True
for key, value in system.items():
    if key % 2 != 0:
        if is_first:
            equations.append(system[key][1])
            is_first = False
            continue
        equations.append(system[key][0])
        equations.append(system[key][1])
    else:
        equations.append(system[key][0])
for eq in range(len(equations)):
    equations[eq] = Eq(equations[eq], 0)

fields_array = []

for key, value in fields.items():
    if key % 2 != 0:
        fields_array.append(value[0])
        fields_array.append(value[1])
    else:
        fields_array.append(value[0])

fields_array.pop()
fields_array.pop()
fields_array = fields_array[::-1]


current_field_solution = 0

current_field_index = 0
current_field = fields_array[current_field_index]

eq = 0
field_solution = 0
enough = 0 
while eq < len(equations):
    if current_field_index == 0:
        current_field_solution = solve(equations[eq], current_field)[0]
        current_field_index +=1
        eq+=1
        continue

    elif current_field_index == 1 or current_field_index == 2:
        equations[eq] = equations[eq].subs(current_field, current_field_solution)
        current_field = fields_array[current_field_index]
        
        current_field_solution = solve(equations[eq], current_field)[0]

       
        current_field_index+=1
        eq+=1
    elif enough == 1:
            break
    
    elif eq < len(equations)-1:
        enough = 1
        equations[eq] = expand(equations[eq]).doit()
        equations[eq-1] = expand(equations[eq-1]).doit()


        current_solution_partial = solve(equations[eq], diff(current_field, x))[0]
        equations[eq-1] = expand(equations[eq-1].subs(diff(current_field, x), current_solution_partial)).doit()
        solution_to_replace_in_partial = solve(equations[eq-1], fields_array[current_field_index])[0]
        current_solution_partial = current_solution_partial.subs(fields_array[current_field_index], solution_to_replace_in_partial)
        solved_partial = integrate(current_solution_partial, x)
        equation_to_solve = equations[eq-1].subs(fields_array[current_field_index-1], solved_partial)
        field_solution = solve(equation_to_solve, fields_array[current_field_index])[0]
        eq+=1

        # if eq < len(equations)-2:
        #     current_field = fields_array[current_field_index]
        #     equations[eq] = expand(equations[eq].subs(fields_array[current_field_index], field_solution)).doit()
        #     current_field_index+=1
        #     current_field = fields_array[current_field_index]
        #     current_field_solution = solve(equations[eq], current_field)[0]
            
        #     eq+=1

        #     current_field = fields_array[current_field_index]
        #     equations[eq] = equations[eq].subs(current_field, current_field_solution)
        #     current_field_index+=1
        #     current_field = fields_array[current_field_index]
        #     current_field_solution = solve(equations[eq], current_field)[0]

equations[eq] = expand(equations[eq].subs(fields_array[current_field_index], field_solution)).doit()
current_field_index+=1
current_field = fields_array[current_field_index]
current_field_solution = solve(equations[eq], current_field)[0]

eq+=1

current_field_index+=1
equations[eq] = expand(equations[eq].subs(current_field, current_field_solution)).doit()
current_field = fields_array[current_field_index]
current_field_index+=1

eq+=1

equations[eq] = expand(equations[eq]).doit()
equations[eq-1] = expand(equations[eq-1]).doit()

current_solution_partial = solve(equations[eq], diff(current_field, x))[0]
equations[eq-1] = expand(equations[eq-1].subs(diff(current_field, x), current_solution_partial)).doit()
solution_to_replace_in_partial = solve(equations[eq-1], fields_array[current_field_index])[0]
current_solution_partial = current_solution_partial.subs(fields_array[current_field_index], solution_to_replace_in_partial)
display(current_solution_partial)
solved_partial = integrate(current_solution_partial, x)
display(solved_partial)
equation_to_solve = equations[eq-1].subs(fields_array[current_field_index-1], solved_partial)
field_solution = solve(equation_to_solve, fields_array[current_field_index])[0]


eq+=1
thisequation = expand(equations[eq].subs(fields_array[current_field_index], field_solution)).doit() 
display(thisequation)



3*v0(x, t)**3*Derivative(v0(x, t), x)/2 - 3*v0(x, t)**2*Derivative(v0(x, t), (x, 2))/4 - 3*v0(x, t)*Derivative(v0(x, t), x)**2/2 - v0(x, t)*Derivative(v0(x, t), (x, 3))/4 + Derivative(v0(x, t), (x, 4))/8

-(Integral(12*v0(x, t)*Derivative(v0(x, t), x)**2, x) + Integral(2*v0(x, t)*Derivative(v0(x, t), (x, 3)), x) + Integral(6*v0(x, t)**2*Derivative(v0(x, t), (x, 2)), x) + Integral(-12*v0(x, t)**3*Derivative(v0(x, t), x), x) + Integral(-Derivative(v0(x, t), (x, 4)), x))/8

Eq(15*v0(x, t)**4*Derivative(v0(x, t), x)/8 + 3*v0(x, t)**2*Derivative(v0(x, t), x)**2/4 - 5*v0(x, t)**2*Derivative(v0(x, t), (x, 3))/8 - 9*v0(x, t)*Derivative(v0(x, t), x)*Derivative(v0(x, t), (x, 2))/4 - Derivative(v0(x, t), t) - 3*Derivative(v0(x, t), x)**3/4 - 3*Derivative(v0(x, t), x)*Integral(v0(x, t)*Derivative(v0(x, t), x)**2, x)/2 - Derivative(v0(x, t), x)*Integral(v0(x, t)*Derivative(v0(x, t), (x, 3)), x)/4 - 3*Derivative(v0(x, t), x)*Integral(v0(x, t)**2*Derivative(v0(x, t), (x, 2)), x)/4 + Derivative(v0(x, t), (x, 5))/16, 0)

In [118]:
zz = Function('zz')(x, t)

expr = zz * diff(zz, x, 3)

step1 = (
    zz * diff(zz, x, 2)
    - integrate(diff(zz, x) * diff(zz, x, 2), x)
)

display(step1)

step2 = step1.subs(
    integrate(diff(zz, x) * diff(zz, x, 2), x),
    Rational(1,2) * diff(zz, x)**2
)

display(step2)


zz(x, t)*Derivative(zz(x, t), (x, 2)) - Derivative(zz(x, t), x)**2/2

zz(x, t)*Derivative(zz(x, t), (x, 2)) - Derivative(zz(x, t), x)**2/2